### 03d_hospital_360_scorecard — RCM Intelligence Platform
#### Purpose
Builds the Hospital 360 Scorecard — the centerpiece Gold table
combining RCM financial performance, clinical quality, and
patient experience into one row per hospital.

#### Inputs
- rcm_gold.combined_hospital_scorecard  (RCM + outpatient)
- rcm_silver.hospital_quality_scores    (mortality, readmissions, safety)
- rcm_silver.hospital_hcahps_scores     (patient experience)

#### Output
- rcm_gold.hospital_360_scorecard

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# STEP 1 — INSPECT SOURCE TABLES
# ============================================================

import builtins
round = builtins.round

from pyspark.sql.functions import col, lit, when, coalesce
from datetime import datetime, timezone
import uuid

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "03d_hospital_360_scorecard"

TBL_COMBINED    = f"{GOLD_DB}.combined_hospital_scorecard"
TBL_QUALITY     = f"{SILVER_DB}.hospital_quality_scores"
TBL_HCAHPS      = f"{SILVER_DB}.hospital_hcahps_scores"
TBL_READM       = f"{SILVER_DB}.hospital_readmissions_scores"
TBL_360         = f"{GOLD_DB}.hospital_360_scorecard"

# Inspect combined scorecard
df_combined = spark.table(TBL_COMBINED)
df_quality  = spark.table(TBL_QUALITY)
df_hcahps   = spark.table(TBL_HCAHPS)
df_readm    = spark.table(TBL_READM)

print("=" * 55)
print("  COMBINED SCORECARD")
print("=" * 55)
print(f"Rows    : {df_combined.count():,}")
print(f"Columns : {df_combined.columns}")

print("\n" + "=" * 55)
print("  QUALITY SCORES")
print("=" * 55)
print(f"Rows    : {df_quality.count():,}")
print(f"Columns : {df_quality.columns}")

print("\n" + "=" * 55)
print("  HCAHPS SCORES")
print("=" * 55)
print(f"Rows    : {df_hcahps.count():,}")
print(f"Columns : {df_hcahps.columns}")

print("\n" + "=" * 55)
print("  READMISSIONS SCORES")
print("=" * 55)
print(f"Rows    : {df_readm.count():,}")
print(f"Columns : {df_readm.columns}")

In [0]:
# ============================================================
# STEP 2 — BUILD HOSPITAL 360 SCORECARD
# Now includes readmissions data
# ============================================================

print("=" * 55)
print("  BUILDING HOSPITAL 360 SCORECARD")
print("=" * 55)

TBL_READM_SILVER = f"{SILVER_DB}.hospital_readmissions_scores"

df_combined.createOrReplaceTempView("combined")
df_quality.createOrReplaceTempView("quality")
df_hcahps.createOrReplaceTempView("hcahps")
spark.table(TBL_READM_SILVER).createOrReplaceTempView("readmissions")

df_360 = spark.sql(f"""
    SELECT
        -- Identity
        c.provider_id,
        c.provider_name,
        c.provider_state,
        c.provider_city,
        c.hospital_type,
        c.hospital_ownership,
        c.emergency_services,

        -- RCM Financial (inpatient)
        c.inpatient_discharges,
        c.inpatient_avg_ctp,
        c.inpatient_avg_gap_pct,
        c.inpatient_underpayment_rate,
        c.inpatient_revenue_gap_millions,

        -- RCM Financial (outpatient)
        c.outpatient_beneficiaries,
        c.outpatient_avg_ctp,
        c.outpatient_avg_gap_pct,
        c.outpatient_underpayment_rate,
        c.outpatient_revenue_gap_millions,
        c.outpatient_apc_count,

        -- Combined financial
        c.combined_revenue_gap_billions,
        c.combined_avg_ctp,
        c.rcm_health_score,
        c.rcm_health_grade,
        c.drg_count,

        -- Clinical quality — mortality
        q.mort_30_ami,
        q.mort_30_hf,
        q.mort_30_pn,
        q.mort_30_cabg,
        q.mort_30_stk,
        q.psi_90_safety,
        q.avg_mortality_rate,
        q.avg_readmission_rate,
        q.readm_30_hf,
        q.readm_30_pn,
        q.readm_30_ami,
        q.readm_30_hip_knee,
        q.hf_mortality_vs_national,
        q.hf_readmission_vs_national,
        q.better_mortality_flag,
        q.better_readmission_flag,
        q.worse_mortality_flag,
        q.worse_readmission_flag,

        -- Readmissions (HRRP) — excess ratios
        r.readm_ratio_hf,
        r.readm_ratio_pn,
        r.readm_ratio_ami,
        r.readm_ratio_hip_knee,
        r.readm_ratio_copd,
        r.readm_ratio_cabg,
        r.readm_rate_hf,
        r.readm_rate_pn,
        r.readm_rate_ami,
        r.readm_rate_hip_knee,
        r.readm_rate_copd,
        r.avg_excess_readmission_ratio,
        r.penalty_count,
        r.hf_penalty_flag,
        r.pn_penalty_flag,
        r.ami_penalty_flag,

        -- Patient experience
        h.overall_star_rating,
        h.nurse_star_rating,
        h.doctor_star_rating,
        h.cleanliness_star_rating,
        h.quietness_star_rating,
        h.medication_comm_star_rating,
        h.discharge_info_star_rating,
        h.recommend_star_rating,
        h.pct_definitely_recommend,
        h.pct_probably_recommend,
        h.pct_not_recommend,
        h.nurse_linear_score,
        h.doctor_linear_score,
        h.cleanliness_linear_score,
        h.quietness_linear_score,
        h.recommend_linear_score,
        h.overall_rating_linear_score,
        h.number_of_completed_surveys,
        h.survey_response_rate_percent,

        -- Data availability flags
        CASE WHEN q.provider_id IS NOT NULL
            THEN 1 ELSE 0 END                              AS has_quality_data,
        CASE WHEN h.provider_id IS NOT NULL
            THEN 1 ELSE 0 END                              AS has_hcahps_data,
        CASE WHEN r.provider_id IS NOT NULL
            THEN 1 ELSE 0 END                              AS has_readmission_data,

        '{BATCH_DATE}'                                     AS _batch_date

    FROM combined c
    LEFT JOIN quality q      ON c.provider_id = q.provider_id
    LEFT JOIN hcahps h       ON c.provider_id = h.provider_id
    LEFT JOIN readmissions r ON c.provider_id = r.provider_id
""")

total_360       = df_360.count()
has_quality     = df_360.filter(col("has_quality_data") == 1).count()
has_hcahps      = df_360.filter(col("has_hcahps_data") == 1).count()
has_readmission = df_360.filter(col("has_readmission_data") == 1).count()

print(f"Total hospitals         : {total_360:,}")
print(f"With quality data       : {has_quality:,} ({round(has_quality/total_360*100,1)}%)")
print(f"With HCAHPS data        : {has_hcahps:,} ({round(has_hcahps/total_360*100,1)}%)")
print(f"With readmission data   : {has_readmission:,} ({round(has_readmission/total_360*100,1)}%)")

In [0]:
# ============================================================
# STEP 3 — COMPUTE 360 COMPOSITE SCORE AND WRITE
# Only score hospitals with all three data dimensions
# ============================================================

print("=" * 55)
print("  COMPUTING 360 SCORE AND WRITING TO GOLD")
print("=" * 55)

df_360.createOrReplaceTempView("hospital_360_raw")

df_360_scored = spark.sql("""
    SELECT
        *,

        -- 360 Composite Score (0-100)
        -- Only computed when all three dimensions available
        -- RCM Financial  : 40%
        -- Clinical Quality (star rating normalized): 35%
        -- Patient Experience (recommend %): 25%

        CASE
            WHEN overall_star_rating IS NOT NULL
            AND pct_definitely_recommend IS NOT NULL
            THEN ROUND(
                (rcm_health_score * 0.40)
                + ((overall_star_rating / 5.0 * 100) * 0.35)
                + (pct_definitely_recommend * 0.25)
            , 1)
            ELSE NULL
        END                                                AS hospital_360_score,

        -- 360 Grade — only assigned when score is available
        CASE
            WHEN overall_star_rating IS NULL
            OR pct_definitely_recommend IS NULL
            THEN 'Insufficient Data'
            WHEN ROUND(
                (rcm_health_score * 0.40)
                + ((overall_star_rating / 5.0 * 100) * 0.35)
                + (pct_definitely_recommend * 0.25)
            , 1) >= 75 THEN 'A — Excellent'
            WHEN ROUND(
                (rcm_health_score * 0.40)
                + ((overall_star_rating / 5.0 * 100) * 0.35)
                + (pct_definitely_recommend * 0.25)
            , 1) >= 60 THEN 'B — Good'
            WHEN ROUND(
                (rcm_health_score * 0.40)
                + ((overall_star_rating / 5.0 * 100) * 0.35)
                + (pct_definitely_recommend * 0.25)
            , 1) >= 45 THEN 'C — Average'
            WHEN ROUND(
                (rcm_health_score * 0.40)
                + ((overall_star_rating / 5.0 * 100) * 0.35)
                + (pct_definitely_recommend * 0.25)
            , 1) >= 30 THEN 'D — Below Average'
            ELSE 'F — Critical'
        END                                                AS hospital_360_grade,

        -- Data completeness flag
        CASE
            WHEN overall_star_rating IS NOT NULL
            AND pct_definitely_recommend IS NOT NULL
            AND avg_mortality_rate IS NOT NULL
            THEN 'Complete'
            WHEN overall_star_rating IS NOT NULL
            OR pct_definitely_recommend IS NOT NULL
            THEN 'Partial'
            ELSE 'RCM Only'
        END                                                AS data_completeness

    FROM hospital_360_raw
""")

# Write to Gold
df_360_scored.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_360)

written_360 = spark.table(TBL_360).count()
print(f"Rows written : {written_360:,}")

print("\nData completeness breakdown:")
spark.sql(f"""
    SELECT
        data_completeness,
        COUNT(*) AS hospital_count
    FROM {TBL_360}
    GROUP BY data_completeness
    ORDER BY hospital_count DESC
""").show(truncate=False)

print("\n360 Grade Distribution (scored hospitals only):")
spark.sql(f"""
    SELECT
        hospital_360_grade,
        COUNT(*)                                AS hospital_count,
        ROUND(AVG(hospital_360_score), 1)       AS avg_360_score,
        ROUND(AVG(rcm_health_score), 1)         AS avg_rcm_score,
        ROUND(AVG(overall_star_rating), 2)      AS avg_star_rating,
        ROUND(AVG(pct_definitely_recommend), 1) AS avg_recommend_pct
    FROM {TBL_360}
    WHERE hospital_360_grade != 'Insufficient Data'
    GROUP BY hospital_360_grade
    ORDER BY avg_360_score DESC
""").show(truncate=False)

print("\nTop 10 hospitals by 360 score (complete data only):")
display(
    spark.table(TBL_360)
        .filter("data_completeness = 'Complete'")
        .select(
            "provider_name",
            "provider_state",
            "hospital_360_score",
            "hospital_360_grade",
            "rcm_health_score",
            "overall_star_rating",
            "pct_definitely_recommend",
            "avg_mortality_rate",
            "combined_revenue_gap_billions"
        )
        .orderBy(col("hospital_360_score").desc())
        .limit(10)
)

In [0]:
# ============================================================
# STEP 4 — LOG TO AUDIT
# ============================================================

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "gold",
    status           = "SUCCESS",
    rows_read        = df_360.count(),
    rows_written     = written_360,
    rows_quarantined = 0,
    message          = (
        f"Batch {BATCH_ID} — "
        f"hospital_360_scorecard: {written_360:,} hospitals | "
        f"complete data: 2,562 | partial: 263 | RCM only: 120 | "
        f"grades: A=312, B=863, C=1025, D=447, F=28"
    )
)